# N-BEATS — Walmart Store Sales Forecasting

**Architecture:** N-BEATS (Neural Basis Expansion Analysis for Time Series). A deep
stack of fully-connected blocks. Each block predicts a *backcast* (reconstruction of
the input) and a *forecast*; the residual is passed to the next block. It is
univariate (past sales only) but much more expressive than DLinear.

Two flavours are compared here:
- **Interpretable** stacks (`trend` + `seasonality`) — baseline, WMAE 2,768.
- **Generic** stacks (`identity`) — tuning, WMAE 2,193 (winner).

**Data view:** long format (`unique_id`, `ds`, `y`), from `src/features/nn_preprocessing.py`.
**Evaluation:** WMAE on the validation period (holiday weeks weighted 5x).
**MLflow structure:** experiment `NBEATS_Training` with runs `NBEATS_Data_Prep`,
`NBEATS_Baseline`, `NBEATS_Tuning`, `NBEATS_Best_Pipeline`.

In [ ]:
# Run from the project root so data/ and src/ resolve, whether launched
# from the repo root or from the notebooks/ folder.
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

In [ ]:
import warnings
import numpy as np
import pandas as pd
import mlflow
import dagshub

from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS
from neuralforecast.losses.pytorch import MAE

from src.features.nn_preprocessing import build_long_df, attach_static, temporal_split, FREQ
from src.evaluation.metrics import calculate_wmae

warnings.filterwarnings("ignore")

# Experiment tracking: DagsHub-hosted MLflow.
dagshub.init(repo_owner="GigiSichinava", repo_name="Walmart-Sales-Forecasting", mlflow=True)
mlflow.set_experiment("NBEATS_Training")

SPLIT_DATE = "2012-01-01"
SEED = 42
print("Tracking URI:", mlflow.get_tracking_uri())

In [ ]:
train_raw = pd.read_csv("data/train.csv")
stores = pd.read_csv("data/stores.csv")
features = pd.read_csv("data/features.csv")
print("train:", train_raw.shape, "| stores:", stores.shape, "| features:", features.shape)

## Run 1 — Data preparation

Same long-format preparation as the other neural models, logged as its own run.

In [ ]:
with mlflow.start_run(run_name="NBEATS_Data_Prep"):
    # Step 1: build the long panel and split by date.
    Y_df = build_long_df(train_raw, stores, features, fill_gaps=True)
    train_df, valid_df, horizon = temporal_split(Y_df, SPLIT_DATE)

    # Step 2: log the preprocessing choices.
    mlflow.log_param("freq", FREQ)
    mlflow.log_param("split_date", SPLIT_DATE)
    mlflow.log_param("gap_fill", "reindex_weekly_y0")
    mlflow.log_param("n_series", int(Y_df["unique_id"].nunique()))
    mlflow.log_param("horizon_weeks", int(horizon))
    mlflow.log_metric("train_rows", float(len(train_df)))
    mlflow.log_metric("valid_rows", float(len(valid_df)))

    print("series:", Y_df["unique_id"].nunique(), "| horizon (weeks):", horizon)

## WMAE evaluation helper

In [ ]:
def evaluate_forecast(forecast_df, valid_df, model_col="NBEATS"):
    # Join forecasts to the true validation values on series id and date.
    merged = forecast_df.merge(
        valid_df[["unique_id", "ds", "y", "IsHoliday"]],
        on=["unique_id", "ds"], how="inner",
    )
    # Neural models can output small negatives; sales can't be negative, so clip.
    preds = merged[model_col].clip(lower=0)
    wmae = calculate_wmae(merged["y"], preds, merged["IsHoliday"])
    return wmae, merged

## Run 2 — Baseline N-BEATS (interpretable stacks)

Trend + seasonality stacks. N-BEATS is heavier than DLinear, so on CPU this run takes
longer. **Result: WMAE 2,768.**

In [ ]:
with mlflow.start_run(run_name="NBEATS_Baseline"):
    params = dict(h=horizon, input_size=52, max_steps=500,
                  stack_types=["identity", "trend", "seasonality"], n_blocks=[1, 1, 1],
                  scaler_type="robust", start_padding_enabled=True,
                  random_seed=SEED, alias="NBEATS")
    mlflow.log_params({k: str(v) for k, v in params.items()})

    nf = NeuralForecast(models=[NBEATS(loss=MAE(), **params)], freq=FREQ)
    nf.fit(df=train_df[["unique_id", "ds", "y"]])

    fcst = nf.predict()
    wmae, _ = evaluate_forecast(fcst, valid_df, "NBEATS")
    mlflow.log_metric("WMAE", float(wmae))
    print(f"NBEATS interpretable WMAE: {wmae:,.2f}")

## Run 3 — Tuning (generic stacks)

Generic `identity` stacks with more blocks — the flexible, less-structured variant.
**Result: WMAE 2,193 — better than the interpretable baseline, and the best neural
model so far.**

In [ ]:
with mlflow.start_run(run_name="NBEATS_Tuning"):
    params = dict(h=horizon, input_size=52, max_steps=1000,
                  stack_types=["identity", "identity"], n_blocks=[3, 3],
                  scaler_type="robust", start_padding_enabled=True,
                  random_seed=SEED, alias="NBEATS")
    mlflow.log_params({k: str(v) for k, v in params.items()})

    nf = NeuralForecast(models=[NBEATS(loss=MAE(), **params)], freq=FREQ)
    nf.fit(df=train_df[["unique_id", "ds", "y"]])

    fcst = nf.predict()
    wmae, _ = evaluate_forecast(fcst, valid_df, "NBEATS")
    mlflow.log_metric("WMAE", float(wmae))
    print(f"NBEATS generic WMAE: {wmae:,.2f}")

## Run 4 — Best pipeline (retrain on all data, save for inference)

The generic config won (2,193 vs 2,768), so `BEST` is the generic stacks. Retrain on
the full history and save for the inference step.

In [9]:
# Best config = the generic stacks (lower WMAE).
BEST = dict(h=horizon, input_size=52, max_steps=1000,
            stack_types=["identity", "identity"], n_blocks=[3, 3],
            scaler_type="robust", start_padding_enabled=True,
            random_seed=SEED, alias="NBEATS")

with mlflow.start_run(run_name="NBEATS_Best_Pipeline"):
    mlflow.log_params({k: str(v) for k, v in BEST.items()})

    # Retrain on the full history so the saved model uses every available week.
    nf_best = NeuralForecast(models=[NBEATS(loss=MAE(), **BEST)], freq=FREQ)
    nf_best.fit(df=Y_df[["unique_id", "ds", "y"]])

    save_path = "saved_models/nbeats_nf"
    os.makedirs(save_path, exist_ok=True)
    nf_best.save(path=save_path, overwrite=True, save_dataset=False)
    mlflow.log_artifacts(save_path, artifact_path="nbeats_nf")
    print("Saved NBEATS model to", save_path)

Seed set to 42
GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 5.2 M  | train
-------------------------------------------------------
5.2 M     Trainable params
0         Non-trainable params
5.2 M     Total params
20.732    Total estimated model params size (MB)
58        Modules in train mode
0         Modules in eval mode


Epoch 0:  95%|█████████▌| 100/105 [00:42<00:02,  2.33it/s, v_num=16, train_loss_step=37.40]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1:  90%|█████████ | 95/105 [00:45<00:04,  2.09it/s, v_num=16, train_loss_step=23.70, train_loss_epoch=77.60] 
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 2:  86%|████████▌ | 90/105 [00:43<00:07,  2.08it/s, v_num=16, train_loss_step=17.50, train_loss_epoch=161.0]   
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 3:  81%|████████  | 85/105 [00:33<00:07,  2.54it/s, v_num=16, train_loss_step=44.20, train_loss_epoch=62.20]  
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 4:  76%|███████▌  | 80/105 [00:30<00:09,  2.60it/s, v_num=16, train_loss_step=19.70, train_loss_epoch=68.90] 
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/

`Trainer.fit` stopped: `max_steps=1000` reached.


Epoch 9: 100%|██████████| 55/55 [00:25<00:00,  2.17it/s, v_num=16, train_loss_step=15.70, train_loss_epoch=92.30]
Saved NBEATS model to saved_models/nbeats_nf
🏃 View run NBEATS_Best_Pipeline at: https://dagshub.com/GigiSichinava/Walmart-Sales-Forecasting.mlflow/#/experiments/1/runs/aded3983cb094ba8a23133edd7ef5cb1
🧪 View experiment at: https://dagshub.com/GigiSichinava/Walmart-Sales-Forecasting.mlflow/#/experiments/1


## Notes

- N-BEATS is univariate (no exogenous features), like DLinear.
- Generic stacks (2,193) beat interpretable stacks (2,768) on this data.
- Best neural result so far, ahead of DLinear (2,679).